Grundregeln:
- jede python Klasse hat eine übergeordnete Aufgabe
- jede python Klasse hält nur einen data file
- jede python Klasse importiert nur dann andere Klassen, wenn sie nur durch diese Importe als Gesamtes funktionieren (das ist in den meisten Fällen der DataManager). Wenn eine andere Klasse kein inhärenter Teil ist, wird sie nur als Funktions-Input für die Methode nutzbar gemacht.
- kein cleaning! Statt stillem Bereinigen stoppt die Funktion bei Ungereimtheiten und zeigt eine Fehlermeldung an

Zu adressierende Probleme:
- predicate requisite kleiner Teil der RO-Relationen. Reicht der aus?
- nur Abstracts in pubmed. Volltext-Reviews / Metaanalysen oder Bücher besser?
- manche Attribute sind bei fast jeder Entität TRUE. Sollte man diese entfernen? >90% TRUE?
- Scalability probleme vor allem der Normalisierung und fill_nan
- Vektordatenbank statt Pubmed_Abstracts_df besser?
- In den manuellen Abfragen wird häufig die selbe Entität in anderer Schreibweise angeboten. Antwort: Erst Mapping, dann Cleaning. 




Adressierte Probleme:
- LLM hat viele falsche Entitäten und Attribute extrahiert. Antwort: StoreCleaner entfernt nach Keywords alle falschen
- Keyword-alleinige Entfernung teils falsch. Antwort: User input kategorisiert in blacklist oder whitelist. 
- Mapping aller Entitäten am Ende wahrscheinlich überfordernd. Antwort: Mapping nach 1000 neuen Entitäten. 
- Mapping nutzt falsche Standardterme. Antwort: Auch hier User input, der kategorisiert und blockiert. 
- LLM kosten sind explodiert, weil 120s von async gewartet wurde, die Antowrt brauchte aber 180s: geändert zu 240s Wartezeit
- LLM_ Input tokens sind explodiert wegen integration aller genutzten Targets und Attributes: Antwort: pruning auf die 300 meistgenutzten Targets
- causally_related ist oft falsch verknüpft. Aus der Liste der Prädikate entfernen? Antwort: Entfernt worden
- bis LLMClient().extract_entities werden nur Attribute ausgefüllt, die TRUE sind. Das ist zu ungenau. Von Anfang an müssen Attribute auch False gelabelt werden. Antwort: Extract entities labelt auch Falsche Entitäten
- mehrere Klassen haben durch das LLM denselben Namen bekommen, was verboten ist. Antwort: Prompt ist angepasst worden
- Abstracts enthalten sehr viele Entitäten, die nichts mit Wunden zu tun haben. Antwort: LLM prompt wurde so angepasst, dass nur domänenspezifische Entitäten integriert werden.
- Die formale Konzeptanalyse hält sich nicht an die BFO/OBO-Architektur. Antwort: attributes haben die form predicate: target. präformierte attributes sind die bfo_classes als is_a:bfo_class um die Einteilung in die BFo-Architektur zu erzwingen.
- es kann mehrere synonme Objekte oder Attribute geben. Antwort: objects und attributes werden per LLM zu standardbegriffen gemappt, dann grouping durch hard code
- Klassennamen werden aus Attributen gebildet. Antwort: Semantische Klassennamen kreiert LLM
- zu viele concepts und attribute? unschaffbar für hard code? erweiterter RAM und TPU
- zu viele Anfragen an LLM? Antwort: batching
- zu komplexe Anfragen an LLM? Antwort: Lange Frage besser als lange Antwort! (gemini 3 flash schafft 1mio tokens)
- komplette Nachverfolgbarkeit für selektives Löschen query-weise oder datumsweise bei schlechten einzelnen Ergebnissen: Antwort: meta_ids
- kein überladen des Speicher: Antwort: immer nur 10 dateien eines Präfix im Speicher und purge method des DataManager()
- LLM-Anfragen dauern zu lange, teils nur eine Anfrage pro 2,5 Minuten. Antwort: asyncio parallel prompting


# Setup

In [ ]:
import nest_asyncio
import torch
nest_asyncio.apply()

import pandas as pd
import sys
from pathlib import Path
from dotenv import load_dotenv

In [ ]:
# Works whether VS Code runs the notebook from project root or notebooks/
_cwd = Path().resolve()
_project_root = _cwd if (_cwd / "config").exists() else _cwd.parent
sys.path.insert(0, str(_project_root))
load_dotenv(_project_root / ".env")

from config.settings import (
    BASE_PATH,
    OWL_OUTPUT_FILE,
    ENTREZ_EMAIL,
    BATCH_SIZE,
    PUBMED_SEARCH_TERMS,
    ALLOWED_PREDICATES,
)
print(f"Project root: {_project_root}")
print(f"ALLOWED_PREDICATES: {ALLOWED_PREDICATES}")

In [ ]:
from wound_ontology.orchestrator import Orchestrator

# Pipeline

## 1. Initialise

In [ ]:
pipeline = Orchestrator(
    entrez_email=ENTREZ_EMAIL,
    allowed_predicates=ALLOWED_PREDICATES,
    base_path=BASE_PATH,
)

## 2. Fetch abstracts

In [ ]:
#pipeline.run_pubmed_search(PUBMED_SEARCH_TERMS)

## 2b. Validate PubMed queries

Fetches a small sample of abstracts for each query in PUBMED_SEARCH and asks
Claude Haiku (via OpenRouter) whether the results are within wound-ontology scope.
Failing queries are auto-revised up to max_retries times.

Requires OPENROUTER_API_KEY in .env.

In [ ]:
# Validate and optionally auto-revise queries via Claude Haiku.
# queries defaults to config.pubmed_query_list.PUBMED_SEARCH.
# out_path writes a JSON report; set to None to skip.
#pipeline.run_validate_pubmed_queries(
#    sample_n=5,
#    max_retries=2,
#    out_path="data/cache/pubmed_query_validation.json",
#)

## 3. Extract entities → ConceptStore

Noise terms (units, pure symbols) are filtered automatically during extraction.

In [ ]:
#pipeline.run_extract_to_store(batch_size=BATCH_SIZE, max_vocab_hints=300)

## 4. Build term mapping

In [ ]:
#pipeline.run_build_term_mapping()

## 5. Review mapping

Inspect and curate auto-merged clusters and suspicious mappings before applying.

In [ ]:
#pipeline.review_mapping()

## 5b. Validate term mapping (SapBERT QC)

Re-embeds every non-identity mapping with SapBERT and flags entries
where cosine similarity < 0.95 — potential false merges to review
before applying the mapping to the store.

In [ ]:
# Returns list of flagged dicts: {term, canonical, cosine, string_sim}
flagged = pipeline.check_term_mapping(threshold=0.95)
print(f"{len(flagged)} suspicious mapping(s) found")

## 6. Apply mapping

In [ ]:
#pipeline.run_apply_mapping()

## 7. Review lists

SapBERT scores every entity against the blocklist:
- cos ≥ 0.92 → auto-block
- cos < 0.40 → silently skip (in scope)
- in between → manual review

Also reviews blocked and whitelisted entries.

In [ ]:
#pipeline.review_lists(llm_batch_size=100, save_skipped=True)

## 7b. Inspect / rewrite blocklist clusters

Groups METHODOLOGY_BLOCKLIST terms by their cluster rules and optionally
rewrites config/blocklist.py with comment-headed sections.

dry_run=True → summary only (safe to run anytime)
dry_run=False → rewrites config/blocklist.py in place

In [ ]:
# Summary only (no file change):
pipeline.cluster_blocklist(dry_run=True)

# Uncomment to rewrite blocklist.py with cluster-headed sections:
#pipeline.cluster_blocklist(dry_run=False)

## 7c. Promote clusters to whitelist

Move specific SapBERT clusters from METHODOLOGY_BLOCKLIST to WHITELIST.
Clusters are matched by partial substring of their comment-header label.
Default target_labels covers the five clusters identified during initial review.

dry_run=True → print what would be promoted, no files written.
dry_run=False → updates config/blocklist.py and config/whitelist.py in place.

In [ ]:
# Move specific SapBERT clusters from METHODOLOGY_BLOCKLIST to WHITELIST.
# Uses the five clusters identified during initial review by default.
# Pass target_labels=[...] to override.
#pipeline.run_promote_clusters(dry_run=False)

## 8. Filter store

In [ ]:
#pipeline.run_clean_store()

## 8b. Reasoner gate — store relations

ELK over the skeleton TBox plus every extracted relation, as an `∃R.Filler`
restriction on each subject. Catches what the category-pair table cannot:
a relation whose filler makes its subject unsatisfiable against MILD/WONT
disjointness.

Runs **before** the hierarchies, so a bad relation never reaches the FCA
lattice. A relation quarantined at the far end has already shaped the lattice
that produced the `is_a` edges, and that damage outlives the axiom.

Read the counts, not just the verdict: anything the adapter could not type is
a term the reasoner never saw, and "clean" over 3 relations is not clean.
Requires `robot` on PATH — without it the gate reports that it did not run.


In [ ]:
pipeline.run_gate_store_relations()

## 9. Build hierarchies (class / mereological)

In [ ]:
print(list(pipeline.store.entities)[:100])

In [ ]:
pipeline.run_build_hierarchies()

In [ ]:
pipeline.review_mere_relations()

## 9b. Reasoner gate — hierarchy edges

The sibling gate. `is_a` edges arrive from three sources — the curated CSV,
the text, and the FCA lattice — and only the first passes through the
extraction schema at all. This is also the only check that sees same-category
disjointness: `Wound base`, `Wound edge` and `Wound pocket` are all
MaterialEntity yet pairwise disjoint, so an edge making something a subclass
of two of them satisfies every schema check.

Quarantined edges are set to 0.0 in the hierarchy and written to
`output/quarantine/` with the reasoner's own explanation.


In [ ]:
pipeline.run_gate_hierarchy_edges()

## 10. Fill missing entity–attribute pairs

In [ ]:
#pipeline.run_store_fill()

## 11. Formal Concept Analysis

In [ ]:
#pipeline.run_fca(subtree_root="Wound", predicates=["has_quality", "part_of", "has_participant", "has_disposition", "inheres_in"])

## 12. Export → OWL

In [ ]:
#pipeline.export_to_owl()

## 13. Enrich OWL (IRI renaming, BFO/RO import, UMLS synonyms)

In [ ]:
#pipeline.run_curate_owl(
#    input_file=OWL_OUTPUT_FILE,
#    output_file=str(Path(OWL_OUTPUT_FILE).parent / "ontology_enriched.owl"),
#    import_bfo=True,
#    import_ro=True,
#)

## 13c. Ground WOND classes (OntologyMapper — ontogpt)

Places every WOND class inside the BFO hierarchy by adding `rdfs:subClassOf`:
- Deterministic via UPPER_KNOWN dict for known upper-level labels.
- OntoGPT SPIRES (LLM) as fallback for domain classes.

Always uses mode=subclass. Results cached in cache_path (delete to re-run).

In [ ]:
# source_file: raw TTL (e.g. data/raw/ground_truth_2606192019.ttl)
# output_file: destination OWL/TTL; defaults to overwriting source_file
#pipeline.run_replace_upper_stubs(
#    source_file="data/raw/ground_truth_2606192019.ttl",
#    output_file="output/owl/ground_truth.owl",
#    dry_run=False,
#)

## 13c. Ground WOND classes (OntoGPT SPIRES)

Adds rdfs:subClassOf and oboInOwl:hasDbXref to every WOND class:
- Upper classes (WOND_0000433–0000486): deterministic UPPER_KNOWN dict.
- Domain classes (WOND_0000001–0000432): OntoGPT SPIRES via LiteLLM/OpenRouter.

Results are cached in data/cache/grounding_cache.json (delete to re-run).
Use only_upper=True for a fast first pass without LLM calls.

In [ ]:
#pipeline.run_map_ontology(
#    owl_file="output/owl/ground_truth.owl",
#    method="ontogpt",          # always mode=subclass
#    model="openrouter/deepseek/deepseek-v4-flash",
#    cache_path="data/cache/grounding_cache.json",
#    dry_run=False,
#)

## 13d. Replace matched WOND classes (OntologyMapper — lexical)

Exact rdfs:label / hasExactSynonym match against DOID > HP > CHEBI > GO > PATO >
OBI > FMA (default priority).  First match in priority order wins.

mode="replace" swaps the WOND IRI for the external OBO IRI.
mode="xref" keeps the WOND IRI and adds oboInOwl:hasDbXref.
mode="subclass" keeps the WOND IRI and adds rdfs:subClassOf.

onto_dir defaults to IMPORT_ONTO_DIR in config/settings.py.

In [ ]:
# onto_dir defaults to IMPORT_ONTO_DIR from config/settings.py
#pipeline.run_map_ontology(
#    owl_file="output/owl/ground_truth_grounded.owl",
#    output_file="output/owl/ground_truth_replaced.owl",
#    method="lexical",
#    mode="replace",
#    use_synonyms=False,
#    dry_run=False,
#)

## 13e. SapBERT grounding (OntologyMapper — sapbert)

Cosine-similarity matching via SapBERT embeddings.  For each source class the
highest-priority target ontology with a match above threshold wins.

Combine with mode="replace" to swap IRIs or mode="subclass" to add subClassOf.

onto_dir defaults to IMPORT_ONTO_DIR in config/settings.py.

In [ ]:
#pipeline.run_map_ontology(
#    owl_file="output/owl/ground_truth_replaced.owl",
#    output_file="output/owl/ground_truth_mapped.owl",
#    method="sapbert",
#    mode="replace",
#    threshold=0.92,
#    dry_run=False,
#)

In [ ]:
# Patch wrong external IRIs and enrich external classes with labels/definitions
# from local source-ontology files (BFO, COB, IAO, OBI, PATO, GO).
# onto_dir defaults to IMPORT_ONTO_DIR in config/settings.py.
#pipeline.run_enrich_external(
#    input_file=str(Path(OWL_OUTPUT_FILE).parent / 'ground_truth_curated.owl'),
#    output_file=str(Path(OWL_OUTPUT_FILE).parent / 'ground_truth_enriched.owl'),
#    dry_run=False,
#)

## 0. Ground Truth Preparation

Run once to align and extend ground_truth.owl before extraction.
target can be any .json (SNOMED), .obo, or .owl file.

Outputs (timestamped in out_dir):
- alignment_candidates.csv — top-K matches per WOND class
- alignment.sssom.tsv — best mapping per class
- source_extended.owl — ground_truth extended with unmatched target stubs

Unmatched **source** classes (your own WOND classes not found in the target)
are printed to the console — these are alignment gaps to address.

In [ ]:
# Fetch wound-related SNOMED CT concepts from UMLS UTS API
# Requires UMLS_API_KEY in .env — run once, outputs to data/reference/
#
# Full fetch (BFS from seed terms):
#concepts = pipeline.fetch_snomed_concepts(out_dir="data/reference")
#
# Re-apply filters without re-querying the API:
#concepts = pipeline.fetch_snomed_concepts(out_dir="data/reference", prune_only=True)
#
# Rebuild output from BFS cache (e.g. after changing SEED_TERMS):
#concepts = pipeline.fetch_snomed_concepts(out_dir="data/reference", rebuild_from_cache=True)
#
#print(f"Fetched {len(concepts)} wound-related SNOMED CT concepts")

In [ ]:
# Align ground truth to SNOMED CT
#result = pipeline.align_ontology(
#    source_owl="output/owl/ground_truth.owl",
#    target="data/reference/snomed_wound_concepts.json",
#    out_dir="data/reference",
#    method="sapbert+lexical",
#    extend=True,   # mint WOND stubs for unmatched SNOMED concepts
#    enrich=True,   # add xrefs + defs to matched WOND classes
#    dry_run=False,
#)

# Align to any OBO ontology (same call, different target):
#result = pipeline.align_ontology(
#    source_owl="output/owl/ground_truth.owl",
#    target="/path/to/Import-Ontologien/bfo-core.owl",
#    out_dir="data/reference",
#    method="lexical",
#    extend=False, enrich=True,
#)

## Generate UML

Runs pyreverse on the src/wound_ontology package and writes PlantUML diagrams
to docs/uml/.  Only public members are included (--filter-mode PUB_ONLY).

Requires pylint (`pip install pylint`).

## 14. Plot hierarchy

In [ ]:
# Requires pylint to be installed: pip install pylint
# Output goes to docs/uml/ (plantuml *.puml files)
#pipeline.generate_uml(out_dir="docs/uml")

In [ ]:
# Full class hierarchy (interactive pyvis graph)
pipeline.plot_hierarchy()

# Subtree only — uncomment to restrict to a root concept:
# pipeline.plot_hierarchy(root="Wound")

# Mereological hierarchy:
pipeline.plot_hierarchy(hierarchy=pipeline.mere_hier, output_file="mere_hierarchy.html")

# Inspection

In [ ]:
#pipeline.lattice.summary()

# Maintenance

> Run these cells selectively — they modify or delete cached data.

In [ ]:
# Purge a specific cache prefix and re-run from that step
# pipeline.dm.purge_by_prefix("context_matrix", confirm=True)